In [1]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import scipy as scp
import multiprocessing

/Users/andpetrouzeniou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def c_quantile(a, Sigma_Y, n_iter, rand):

    N = len(Sigma_Y)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_Y, n_iter)
    diags = np.tile((1 / np.diagonal(Sigma_Y) ** (1/2)), (n_iter, 1))
    
    draws = abs(draws * diags)
    
    lst = np.amax(draws, axis=1)
    return np.quantile(np.array(lst), a)


In [3]:
def d_quantile(b, k, Sigma_X, n_iter, rand):

    N = len(Sigma_X)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_X, n_iter)
    lst = np.amax(draws, axis=1) - np.amax((-1 * draws), axis=1)

    return np.quantile(np.array(lst), b)


In [23]:
def make_I_list(rank_list, X_temp, delta_L, delta_U):
    
    rank = np.max(np.array(rank_list))
    
    # need matrix A[i,j] = X_temp[j] + delta_L[j,i]
    
    X_temp_mat = np.tile(X_temp, (len(X_temp), 1))
    A = X_temp_mat + np.transpose(delta_L)
    
    np.fill_diagonal(A, np.nan)
    A = A[~np.isnan(A)].reshape(A.shape[0], A.shape[1] - 1)

    row_ranks = np.partition(A, -rank, axis=1)[:, -rank]
    
    indices = np.where(X_temp >= row_ranks, 1, 0)
        
    return indices


In [5]:
def c_2_quantile(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):

    lst = []

    Sigma_Y = Sigma

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        rank = np.max(np.array(rank_list))

        I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)
        
        I_indices = np.where(I_list == 1)[0]    
        lst.append(np.max((abs(X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))
    
#         s = []
                                        
#         for i in range(len(rank_list)):
                                        
#             I = I_list[i]
#             I_indices = np.where(I == 1)[0]
            
#             s.append(np.max((abs(X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))

    return np.quantile(np.array(lst), ab)


In [6]:
def c_2_quantile_asymmetric(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):

    ab_p = 1 - ab
    ab_p = ab_p / 2
    ab_p = 1 - ab_p
    
    lst_l = []
    lst_r = []

    Sigma_Y = Sigma

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        rank = np.max(np.array(rank_list))

        I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)
        
        I_indices = np.where(I_list == 1)[0]    
        lst_r.append(np.max(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))
        lst_l.append(np.min(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))

    return([np.quantile(-1 * np.array(lst_l), ab_p), np.quantile(np.array(lst_r), ab_p)])


In [7]:
def c_3_quantile(ab, subset, Sigma, n_iter, rand):
    
    lst = []
    N = len(Sigma)
    
    for i in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(N), Sigma)
        
        draw = draw[subset]
        diag = np.diagonal(Sigma)[subset]
        lst.append(np.max(abs(draw / (diag ** (1/2)))))
            
    return np.quantile(np.array(lst), ab)


In [8]:
# projection CI

def CI_proj(rank_list, y_vec, a, Sigma_Y, n_iter, quantile_dict, rand):

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])
        
    try:
        c_a = quantile_dict[str(a) + str(Sigma_Y) + "c"]
    except:
        c_a = c_quantile(a, Sigma_Y, n_iter, rand = rand)
        quantile_dict[str(a) + str(Sigma_Y) + "c"] = c_a
        
    CI_list = []
        
    for i in range(len(rank_list)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


In [9]:
def two_step_CI(rank_list, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    k = len(y_vec)

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])

    d = d_quantile(1 - b, k, Sigma_Y, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    L = r - l - d
    U = r - l + d
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile(1-a+b, rank_list, Sigma_Y, L, U, int(n_iter), rand = rand)
    print()
    
    CI_list = []
    
    for i in range(len(rank_list)):
        th_hat = th_hat_lst[i]
        CI_t = [y_vec[th_hat] - bd * (Sigma_Y[th_hat, th_hat] ** 0.5), y_vec[th_hat] + bd * (Sigma_Y[th_hat, th_hat] ** 0.5)]
        CI_list.append(CI_t)

    return CI_list


In [10]:
def two_step_CI_asymmetric(rank_list, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    k = len(y_vec)

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])

    d = d_quantile(1 - b, k, Sigma_Y, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    L = r - l - d
    U = r - l + d
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile_asymmetric(1 - a + b, rank_list, Sigma, L, U, int(n_iter), rand = rand)
    bd_l = bd[0]
    bd_r = bd[1]
    print()
    
    CI_list = []
    
    for i in range(len(rank_list)):
        
        CI_lower_bound = y_vec[th_hat_lst[i]] - bd_r * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + bd_l * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


# Get CSs for all top 50 CZs

In [ ]:
cz_list = ["Port St. Lucie", "Chicago", "Philadelphia", "New York", "Newark", "Milwaukee", "Atlanta", "St. Louis", "Cleveland", "Miami", "Detroit", "Pittsburgh", "Columbus", "Bridgeport", "Dallas", "Washington DC", "Austin", "Raleigh", "Cincinnati", "Charlotte", "Kansas City", "New Orleans", "Nashville", "Baltimore", "Buffalo", "Indianapolis", "Grand Rapids", "Dayton", "San Francisco", "Tampa", "Salt Lake City", "Boston", "Houston", "Phoenix", "Minneapolis", "Denver", "Los Angeles", "Jacksonville", "Fort Worth", "San Antonio", "Orlando", "San Jose", "Sacramento", "San Diego", "Manchester", "Providence", "Fresno", "Las Vegas", "Portland", "Seattle"]

In [ ]:
def make_CIs(cz):

    Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy()
    X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy().flatten()
    Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy()).flatten()
    rank_list = [i for i in range(1,int(len(Y) / 3))]

    print("Projection")
    proj = pd.DataFrame(CI_proj(rank_list, Y, 0.95, Sigma, 10000, {}, rng))

    print("Two Step")
    two_step = pd.DataFrame(two_step_CI(rank_list, Y, 0.05, 0.005, Sigma, 10000, {}, rng))
    
    print("Two Step Asymmetric")
    two_step_asymmetric = pd.DataFrame(two_step_CI_asymmetric(rank_list, Y, 0.05, 0.005, Sigma, 10~000, {}, rng))
    
    proj.to_csv("All_CIs/Projection_"+cz+".csv")
    two_step.to_csv("All_CIs/TwoStep_"+cz+".csv")
    two_step_asymmetric.to_csv("All_CIs/TwoStepAsymmetric_"+cz+".csv")
    
    

In [ ]:
rng = np.random.default_rng(42)

pool = multiprocessing.Pool(55)
p_cov_prob, shaikh_cov_prob, p_length, shaikh_length = zip(*pool.map(make_CIs, cz_list))

In [13]:
import os

notebook_path = os.getcwd()
print(notebook_path)

/Users/andpetrouzeniou/Documents/GitHub/Inference_on_Multiple_Winners/Neighborhood_Effects_Application


# Seattle and Cleveland Urban CZs

In [20]:

Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy()
X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy().flatten()
Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy()).flatten()


In [21]:
print(len(Sigma))

132


In [ ]:
rng = np.random.default_rng(42)

print("Two Step")
two_step_sea = two_step_CI([i for i in range(1,int(len(Y) / 5))], Y, 0.05, 0.005, Sigma, 10000, {}, rng)

print("Projection")
proj_sea = CI_proj([i for i in range(1,int(len(Y) / 5))], Y, 0.95, Sigma, 10000, {}, rng)
    

Two Step

012345678910111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989910010110210310410510610710810911011111211311411511611711811912012112212312412512612712812913013113213313413513613713813914014114214314414514614714814915015115215315415515615715815916016116216316416516616716816917017117217317417517617717817918018118218318418518618718818919019119219319419519619719819920020120220320420520620720820921021121221321421521621721821922022122222322422522622722822923023123223323423523623723823924024124224324424524624724824925025125225325425525625725825926026126226326426526626726826927027127227327427527627727827928028128228328428528628728828929029129229329429529629729829930030130230330430530630730830931031131231331431531631731831932032132232332432532632732832933033133233333433533633733833934034134234334434534634734834935035135235335435535635735835936036136236336436536

In [15]:
print("Projection")
print(proj_sea)
print("Two Step")
print(two_step_sea)

Projection
[[-0.013042651906735425, 0.4303171784610174], [0.03980153356619037, 0.37546421298809163], [0.05124329232907429, 0.34136255422520767], [0.05750516154906621, 0.3065548250052158], [0.012106249308098227, 0.3415676172461838], [0.031639840757622384, 0.32144108579665964], [-0.025029520940763855, 0.3470474874950459], [0.0015596731781280515, 0.316304573376154], [-0.10476856391626158, 0.3961936104705436], [0.01510521750450805, 0.27412804904977395], [-0.042761399695701086, 0.3316972462499831], [0.022986709220501236, 0.26401245733378076], [-0.1561999442047044, 0.4243508107589864], [0.017193207469190133, 0.2507288990850919], [-0.03242204895131251, 0.2942039155055945], [-0.058457463833783946, 0.31709615038806593], [-0.03732573101785949, 0.28856865757214145], [0.02099516791341062, 0.22823313864087136], [-0.07386926684679501, 0.321430653401077], [-0.061845923610159226, 0.3090477901644412], [-0.030483319381783322, 0.2773087259360653], [-0.2607839312424593, 0.5066605377967413], [-0.2126580261

In [10]:

Sigma = pd.read_csv("Neighborhood_Outputs/Sigma_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()
X = pd.read_csv("Neighborhood_Outputs/X_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy().flatten()
Y = np.array(pd.read_csv("Neighborhood_Outputs/Y_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()).flatten()

Sigma = Sigma[len(X):,len(X):]

In [16]:
rng = np.random.default_rng(90)

print("Projection")
proj_cle = CI_proj([i for i in range(1,int(len(Y) / 5))], Y, 0.95, Sigma, 1000, {}, rng)

print("Two Step")
two_step_cle = two_step_CI([i for i in range(1,int(len(Y) / 5))], Y, 0.05, 0.005, Sigma, 1000, {}, rng)
    

Projection
Two Step
01234567891011121314151617181920212223242526272829303132333435363738394041424344454647484950515253545556575859606162636465666768697071727374757677787980818283848586878889909192939495969798991001011021031041051061071081091101111121131141151161171181191201211221231241251261271281291301311321331341351361371381391401411421431441451461471481491501511521531541551561571581591601611621631641651661671681691701711721731741751761771781791801811821831841851861871881891901911921931941951961971981992002012022032042052062072082092102112122132142152162172182192202212222232242252262272282292302312322332342352362372382392402412422432442452462472482492502512522532542552562572582592602612622632642652662672682692702712722732742752762772782792802812822832842852862872882892902912922932942952962972982993003013023033043053063073083093103113123133143153163173183193203213223233243253263273283293303313323333343353363373383393403413423433443453463473483493503513523533543553563573583593603613623

In [17]:
print("Projection")
print(proj_cle)
print("Two Step")
print(two_step_cle)

Projection
[[0.08052945254703175, 0.36228117155311623], [0.08052945254703175, 0.36228117155311623], [0.06257987113040532, 0.2839221729697427], [0.06257987113040532, 0.2839221729697427], [0.07009341526151919, 0.2746891488386288], [0.07009341526151919, 0.2746891488386288], [0.03853699494040504, 0.25221180915974295], [0.03853699494040504, 0.25221180915974295], [0.032480820817694406, 0.23206870328245358], [0.032480820817694406, 0.23206870328245358], [0.053085747226705365, 0.20588287687344262], [0.053085747226705365, 0.20588287687344262], [0.04325198645916224, 0.21149257764098578], [0.04325198645916224, 0.21149257764098578], [0.036882103359231505, 0.2099851007409165], [0.036882103359231505, 0.2099851007409165], [0.039800044232157814, 0.1954673198679902], [0.039800044232157814, 0.1954673198679902], [-0.019291168526054003, 0.23672011262620202], [-0.019291168526054003, 0.23672011262620202], [-0.008507610430401422, 0.21356383453054945], [-0.008507610430401422, 0.21356383453054945], [0.015692972

In [18]:
pd.DataFrame(proj_cl).to_csv("Neighborhood_Outputs/CI_Cleveland.csv")
pd.DataFrame(proj_sea).to_csv("Neighborhood_Outputs/CI_Seattle.csv")

pd.DataFrame(two_step_cl).to_csv("Neighborhood_Outputs/CI_Cleveland.csv")
pd.DataFrame(two_step_sea).to_csv("Neighborhood_Outputs/CI_Seattle.csv")